# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
BASE_API_URL = "http://localhost:11434/v1"
MODEL = 'gpt-oss'
client = OpenAI(base_url=BASE_API_URL, api_key='ollama')

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [7]:
def select_relevant_links(url):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'press release',
   'url': 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html'},
  {'type': 'company page', 'url': 'https://nebula.io/'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [9]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [10]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
deepseek-ai/DeepSeek-V4.1-Flash
Updated
7 days ago
•
366k
•
2.86k
Edge0/Edge0-35B-A3B-preview
Updated
3 days ago
•
27.8k
•
3.1k
m-a-p/YuE2-3B
Updated
about 17 hours ago
•
9.39k
•
642
Qwen/Qwen3.8-27B
Updated
Aug 14
•
7.67M
•
15.4k
openbmb/MiniCPM5-2B
Updated
5 days ago
•
324k
•
1.51k
Browse 2M+ models
Spaces
Running
on
Zero
Agents
353
MiniMax H3 Turbo LoRA
🎬
353
Video generation w

In [20]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt_humorous = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [12]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [13]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ndeepseek-ai/DeepSeek-V4.1-Flash\nUpdated\n7 days ago\n•\n366k\n•\n2.86k\nEdge0/Edge0-35B-A3B-preview\nUpdated\n3 day

In [ ]:
def create_brochure(company_name, url, humorous=False):
    system_prompt = brochure_system_prompt if humorous is False else brochure_system_prompt_humorous
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [17]:
create_brochure("HuggingFace", "https://huggingface.co")

**Hugging Face**  
*The AI community building the future.*

---

## 1.  About Us  

- **Mission** – Empower every AI practitioner by making state‑of‑the‑art models, datasets, and applications easily accessible, collaborative, and ready to deploy.  
- **Vision** – A world where anyone can build, share, and improve AI systems using open, community‑driven tools.  

## 2.  Our Platform  
| Feature | What It Enables |
|---------|-----------------|
| **2M+ Models** | Fast‑track any NLP/vision/workflow with pre‑trained weights. |
| **500k+ Datasets** | Democratise training – just pick a ready‑made data set. |
| **1M+ Spaces** | Deploy and remix ML demos with no infrastructure hassle. |
| **Buckets & Storage** | Store, share, and version large model files and data sets. |
| **Inference Endpoints & Providers** | Scale any model to production with minimal latency. |

All components are open‑source first; we ship fully‑managed SaaS for the enterprise crowd.

---

## 3.  Community & Ecosystem  
- **Open‑source Stack** – Hugging Face Transformers, Tokenizers, Datasets, 🤗 Accelerate, etc.  
- **Collaborative Spaces** – Run, remix, and fork over 1 M AI applications.  
- **Chat & Notes** – HuggingChat, AI Notes – source‑grounded, citation‑aware research aides.  
- **Channels** – Discord, Forum, GitHub, Blog, and a thriving weekly “Daily Papers” roundup.  
- **International Reach** – Multi‑language support, global model collection.

---

## 4.  Enterprise Solutions  
- **Hugging Face PRO** – Premium model hosting, private model registries, and enhanced security.  
- **Enterprise Support** – 24/7 SLAs, dedicated engineers, and audit‑ready compliance.  
- **Inference Providers & Endpoints** – Managed serverless inference, auto‑scaling, and on‑prem options.  
- **Storage Buckets & Versioning** – Unlimited public & private storage, fine‑grained access control.  

Ideal for fintech, healthcare, e‑commerce, and any industry that needs reliable, explainable AI at scale.

---

## 5.  Our Culture  
- **Community‑First** – Decisions come from contributors, researchers, and users worldwide.  
- **Open‑Source Champion** – All core libraries are MIT‑licensed; we value transparency.  
- **Rapid Innovation** – Continuous integration of the latest research papers into pre‑built models.  
- **Inclusive & Diverse** – Active outreach to underrepresented groups in tech.  

---

## 6.  Join the Team  
Hugging Face is growing fast—look for roles like:

- Software Engineer (Python, Rust, Kubernetes)  
- Machine Learning Engineer / Researcher  
- DevOps / Platform Engineer  
- Product & Program Manager  
- Customer Success & Support Engineer  

We thrive on curiosity, deep technical expertise, and a passion for community. If you want to shape the future of AI together, check our Careers page on Hugging Face’s website or send your résumé to hires@huggingface.co.

---

## 7.  Get in Touch

- **Website:** https://huggingface.co  
- **Contact:** contact@huggingface.co  
- **Social:** Discord ‑ #huggingface, GitHub – huggingface 🤗  
- **Newsletter** – Subscribe for weekly insights on cutting‑edge research and product updates.

---

*Hugging Face – where the AI community comes together to build better models, datasets, and applications that power the world of tomorrow.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [26]:
def stream_brochure(company_name, url, humorous=False):
    system_prompt = brochure_system_prompt if humorous is False else brochure_system_prompt_humorous
    stream = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [24]:
stream_brochure("HuggingFace", "https://huggingface.co")

**Hugging Face**  
*The AI community building the future*

---

## About Us  
Hugging Face is the premier collaboration platform for the global machine‑learning community.  
- **2 + million models, 500 k+ datasets, 1 M+ applications** – all freely discoverable, shareable, and reusable.  
- Built on a fully open‑source stack, the platform lets researchers, developers, and enterprises ship models, datasets, and applications faster and with higher quality.  
- Powered by a vibrant community of contributors, forums, Discord, GitHub, and regular newsletters (Daily Papers, Blog Posts).

---

## Our Core Offerings  

| Feature | What It Means |
|---|---|
| **Model Hub** | Upload, share, and version your models. Explore pre‑trained models like DeepSeek‑V4.1‑Flash, Qwen‑3.8‑27B, and many more. |
| **Dataset Hub** | Browse, create, and host millions of public datasets: GLUE, IMDb, SQuAD, and unique community datasets such as `openbmb/UltraData`. |
| **Spaces** | Deploy interactive AI demos (video generation, image editing, music creation) with *Zero* servers—no infrastructure to manage. |
| **Buckets** | Secure, scalable object storage for large files (model weights, corpora). |
| **Inference API** | Instant access to inference endpoints, with flexible pricing and enterprise support. |
| **Enterprise Solutions** | Hugging Face PRO & Enterprise Support, custom inference providers, and dedicated storage solutions for mission‑critical workloads. |

---

## How We Empower Customers  

| Audience | Value Proposition |
|---|---|
| **Research & Academic** | Open access to state‑of‑the‑art models and large-scale datasets; version control across experiments. |
| **Start‑ups & Product Teams** | Rapid prototyping via Spaces, zero‑cost deployment of demos, and paid inference for scale. |
| **Large Enterprises** | Dedicated support, private model hosting, compliance‑ready storage (Buckets), and custom inference pipelines. |
| **Developers & Data Scientists** | Unified API, SDKs (Python, JavaScript), and community tools for end‑to‑end ML workflows. |

---

## Culture & Community  

- **Open Source at heart** – every model, dataset, and toolkit is released under permissive licenses.  
- **Collaboration first** – Git‑based versioning, pull request reviews, and the Hugging Face Forum encourage constructive feedback.  
- **Global, inclusive, and fast** – members from academia, industry, and hobbyists contribute daily; the community fuels rapid iteration.  
- **Learning & Growth** – Hugging Chat, Collections of curated models, Discord discussions, and a rich blog keep members up to speed with the latest in AI research.

---

## Join Us

### Careers  
Hugging Face is continuously looking for passionate engineers, data scientists, community managers, and product experts to shape the next wave of AI.  
> *Explore current openings on our Careers page and be part of a team that is democratizing machine learning worldwide.*

### Get Started  
- **Model**: [🤗 Models](https://huggingface.co/models)  
- **Dataset**: [🤗 Datasets](https://huggingface.co/datasets)  
- **Space**: [🤗 Spaces](https://huggingface.co/spaces)  
- **Enterprise**: [🤗 Enterprise Support](https://huggingface.co/enterprise)  

---  

**Hugging Face – Where Community Meets Innovation.**  
> *Join us in building the future of AI.*

In [27]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co", humorous=True)

# Hugging Face: The (hug‑ing) AI Playground 🎉

## 🚀 About Us

*Founded on the principle that **“AI should be as open as a hug,”** Hugging Face is the *global hive* where you’ll find 2 million+ models, 500 k+ datasets and 1 million+ ready‑to‑use apps.  
We’re the platform that lets the machine‑learning world *collaborate, compete, and make coffee together.”*

- **Open Source Stack** – Your code, your rules, your community.  
- **Zero‑Cost, Unlimited Public Models** – Share, remix, and re‑run with just a click.  
- **Spaces & Buckets** – Deploy, prototype, and store everything from a chatbot to a 3‑D rendering pipeline without the usual infrastructure headaches.  
- **Enterprise‑grade Support** – Hugging Face PRO, inference endpoints, and dedicated team backing for production workloads.  
- **Discord, Forum, GitHub** – We’re everywhere you’re already hanging out. Think of it as the social network that has *actual* AI.  

## 🎭 Our Culture

| 🌈 Identity | 👥 Community | 🕑 Pace | 😹 Humor |
|:-----------:|:------------:|:------:|:--------:|
| **Inclusive & Diverse** – Everyone’s invited to pull up a seat and contribute. | **Collaboration‑First** – “If it’s not on Public, it’s not ours.” | **Rapid‑Iterative** – 30‑second model demos (yes, really). | **Playful** – We joke, we learn, we hack. If you’re not laughing, we’re doing it wrong. |

- **GitHub‑first mindset** – Pull requests are the new handshakes.  
- **“Build‑and‑Learn” days** – You code, you deploy, you celebrate in a Slack‑styled channel.  
- **Open‑Source Fridays** – We release new LLMs, libraries, and even pet project playlists for the community.

## 🎯 Who’s Using Us

| Industry | Where They’re Using Hugging Face |
|:--------:|:-------------------------------:|
| Finance | Generative risk modeling, compliance bots. |
| Healthcare | Medical‑image LLM finetunes, patient‑chat UIs. |
| Entertainment | Video generation, music creation, real‑time translation. |
| E‑Commerce | Personalization engines, inventory forecasting. |
| Academia | Research prototyping, open‑data sharing, education tools. |

> *“We moved from a 3‑node Docker lab to Hugging Face PRO overnight. The time‑to‑deployment dropped by 70% and our interns now get to actually do inference, not just read docs.” – Sam, Enterprise ML Lead*

## 👶 Careers & Jobs

| Role | What You’ll Do | What We’ll Offer |
|------|----------------|-----------------|
| **ML Engineer** | Build & ship production‑grade pipelines. | Cloud credits, equity, “AI‑hug” bonuses. |
| **Research Scientist** | Publish papers, push the model frontier. | Conference support, open‑source swag. |
| **Data Engineer** | Curate 500k+ datasets, maintain data hygiene. | Data‑centric mentorship, hack‑days. |
| **Product Manager** | Shape the next wave of AI products. | Cross‑functional teams, global co‑ops. |
| **Community Lead** | Grow Discord/Forum, organize events. | Creator’s stipend, early‑access to tools. |

*We’re hiring at every level: senior researchers, junior devs, devops, UX designers, and the occasional AI‑literacy evangelist.*

> *Our hiring process <ins>doesn’t</ins> include magic tricks. It’s built on a friendly interview panel, a quick coding challenge, and a conversation about what you’d do if your model could talk.*

## 📑 Why Choose Hugging Face

- **First‑Class Open‑Source** – Own it, fork it, sell it; we just want your ideas to fly.  
- **Zero‑Cost Prototyping** – No hidden charges on your trial runs.  
- **Enterprise‑Grade Support** – From inference endpoints to dedicated R&D partnerships.  
- **Community‑Driven Innovation** – Be the first to see the latest LLMs, embeddings, and AI gadgets.  
- **Work‑Life‑AI Symbiosis** – Our Slack bots are already scheduling your next coffee break.

## 🎉 Join the Hug (and the AI) Revolution

> *“When I first deployed a model on Hugging Face Spaces, I felt the same excitement as discovering the world’s first emoji—small, but absolutely transformative.” – Jenna, 20 yo data scientist*

Whether you’re a **customer** looking to integrate cutting‑edge AI, a **developer** craving an open platform, or a **recruit** itching to jump into a culture that rewards open minds and caffeinated nights, Hugging Face is the place to be.  

**Visit:** https://huggingface.co | **Apply:** https://huggingface.co/jobs | **Connect:** #huggingface on Discord  

_“Because if hugging is a social thing, then hugging AI is the future.”_

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>